# Predict Premier League goalscorers

## Data Acquisition & Feature Engineering

This notebook documents the process of building the dataset used to train a machine learning model for predicting Premier League goalscorers during the 2026/27 season.

The process begins with historical player data and requests to the Fantasy Premier League (FPL) API, followed by data cleaning, merging and feature engineering.

The final output is a structured dataset containing player-level features and goal-scoring outcomes, ready for model training and evaluation.

In [72]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path
import os
from sklearn.preprocessing import OrdinalEncoder

### Player history read

In [3]:
# Import player histories as large dataset

players_dir = Path("data/vaastav/Fantasy-Premier-League-master/data/2025-26/players")
histories = []

for player_dir in players_dir.iterdir():
    if player_dir.is_dir():
        history_file = player_dir/"history.csv"
        
        if not history_file.exists():
            continue
        
        history = pd.read_csv(history_file)

        # Add player name and ID from directory name
        player_id = int(player_dir.name.split("_")[-1])
        player_name = "_".join(player_dir.name.split("_")[:-1])

        history['player_id'] = player_id
        history['player_name'] = player_name

        histories.append(history)

historical_df = pd.concat(histories, ignore_index=True)

In [4]:
historical_df.size

77517

In [5]:
historical_df['player_id'].nunique()

611

In [6]:
historical_df['season_name'].value_counts()

season_name
2024/25    535
2023/24    433
2022/23    349
2021/22    250
2020/21    193
2019/20    151
2018/19    118
2017/18     87
2016/17     62
2015/16     46
2014/15     33
2013/14     27
2012/13     19
2011/12     12
2010/11      9
2008/09      8
2009/10      8
2007/08      6
2006/07      3
Name: count, dtype: int64

In [7]:
historical_df[['player_id', 'player_name', 'season_name',
               'goals_scored', 'expected_goals']].head(20)

,player_id,player_name,season_name,goals_scored,expected_goals
0,116,Aaron_Hickey,2022/23,0,0.38
1,116,Aaron_Hickey,2023/24,0,0.20
2,116,Aaron_Hickey,2024/25,0,0.00
3,674,Aaron_Ramsdale,2016/17,0,0.00
4,674,Aaron_Ramsdale,2017/18,0,0.00
5,674,Aaron_Ramsdale,2019/20,0,0.00
6,674,Aaron_Ramsdale,2020/21,0,0.00
7,674,Aaron_Ramsdale,2021/22,0,0.00
8,674,Aaron_Ramsdale,2022/23,0,0.00
9,674,Aaron_Ramsdale,2023/24,0,0.00


In [8]:
historical_df.to_csv("data/historic_players_256.csv", index=False)

### Import data

In [9]:
player_history245 = pd.read_csv("data/historic_players_256.csv")
match_logs = pd.read_csv("data/vaastav/Fantasy-Premier-League-master/data/2025-26/gws/merged_gw.csv")
fixture_data = pd.read_csv("data/vaastav/Fantasy-Premier-League-master/data/2025-26/fixtures.csv")
team_data = pd.read_csv("data/vaastav/Fantasy-Premier-League-master/data/2025-26/teams.csv")

### Data exploration and cleaning

In [10]:
match_logs.shape

(29757, 46)

In [11]:
match_logs.columns

Index(['name', 'position', 'team', 'xP', 'assists', 'bonus', 'bps',
       'clean_sheets', 'creativity', 'element', 'expected_assists',
       'expected_goal_involvements', 'expected_goals',
       'expected_goals_conceded', 'fixture', 'goals_conceded', 'goals_scored',
       'ict_index', 'influence', 'kickoff_time', 'minutes', 'modified',
       'opponent_team', 'own_goals', 'penalties_missed', 'penalties_saved',
       'red_cards', 'round', 'saves', 'selected', 'starts', 'team_a_score',
       'team_h_score', 'threat', 'total_points', 'transfers_balance',
       'transfers_in', 'transfers_out', 'value', 'was_home', 'yellow_cards',
       'clearances_blocks_interceptions', 'defensive_contribution',
       'recoveries', 'tackles', 'GW'],
      dtype='object')

In [12]:
# Define columns to keep from gw_csv/match_logs dataset
cols_to_keep = ["name",
                "opponent_team",
                "was_home",
                "round",
                "minutes",
                "goals_scored",
                "threat",
                "expected_goals",
                "goals_conceded",
                "expected_goals_conceded",
                "fixture",
                "kickoff_time",
                "position",
                "team",
                "element"]

match_logs = match_logs[cols_to_keep]
match_logs.head()

,name,opponent_team,was_home,round,minutes,goals_scored,threat,expected_goals,goals_conceded,expected_goals_conceded,fixture,kickoff_time,position,team,element
0,Reinildo Mandava,19,True,1,90,0,0.0,0.0,0,0.56,5,2025-08-16T14:00:00Z,DEF,Sunderland,541
1,Lewis Dobbin,15,True,1,0,0,0.0,0.0,0,0.00,2,2025-08-16T11:30:00Z,MID,Aston Villa,57
2,Ryan Christie,12,False,1,0,0,0.0,0.0,0,0.00,1,2025-08-15T19:00:00Z,MID,Bournemouth,87
3,Zeki Amdouni,18,False,1,0,0,0.0,0.0,0,0.00,6,2025-08-16T14:00:00Z,FWD,Burnley,216
4,Lucas Tolentino Coelho de Lima,17,False,1,90,0,4.0,0.0,3,0.68,5,2025-08-16T14:00:00Z,MID,West Ham,612


In [13]:
fixture_data.columns

Index(['code', 'event', 'finished', 'finished_provisional', 'id',
       'kickoff_time', 'minutes', 'provisional_start_time', 'started',
       'team_a', 'team_a_score', 'team_h', 'team_h_score', 'stats',
       'team_h_difficulty', 'team_a_difficulty', 'pulse_id'],
      dtype='object')

In [14]:
""" Define fixture features to keep
Event is gameweek number
id is a unique id for the fixture
team_a and team_h are the team ids for the home and away teams in that fixture
difficulty is determined pre match so there is no data leakage concerns
"""

cols_to_keep = ["event",
                "id",
                "team_a",
                "team_h",
                "team_h_difficulty",
                "team_a_difficulty"
]

fixture_data = fixture_data[cols_to_keep]
fixture_data.head()

,event,id,team_a,team_h,team_h_difficulty,team_a_difficulty
0,1,1,4,12,3,4
1,1,2,15,2,3,4
2,1,3,10,6,3,4
3,1,6,3,18,1,3
4,1,5,19,17,2,3


In [15]:
team_data.columns

Index(['code', 'draw', 'form', 'id', 'loss', 'name', 'played', 'points',
       'position', 'short_name', 'strength', 'team_division', 'unavailable',
       'win', 'link_url', 'strength_overall_home', 'strength_overall_away',
       'strength_attack_home', 'strength_attack_away', 'strength_defence_home',
       'strength_defence_away', 'pulse_id'],
      dtype='object')

In [16]:
# Drop all columns except "name" and "id"

cols_to_keep = ["name",
                "id"
]

team_data = team_data[cols_to_keep]

In [17]:
print(f"Original rows: {len(match_logs):,}")
print(f"Exact duplicates: {match_logs.duplicated().sum():,}")

Original rows: 29,757
Exact duplicates: 10


In [18]:
duplicates_df = match_logs[match_logs.duplicated(keep=False)]
duplicates_df

,name,opponent_team,was_home,round,minutes,goals_scored,threat,expected_goals,goals_conceded,expected_goals_conceded,fixture,kickoff_time,position,team,element
7,Ben Gannon-Doak,4,True,1,0,0,0.0,0.00,0,0.00,1,2025-08-15T19:00:00Z,MID,Liverpool,391
91,Junior Kroupi,12,False,1,1,0,0.0,0.00,1,0.11,1,2025-08-15T19:00:00Z,FWD,Bournemouth,100
291,Ben Gannon-Doak,4,True,1,0,0,0.0,0.00,0,0.00,1,2025-08-15T19:00:00Z,MID,Liverpool,391
638,Junior Kroupi,12,False,1,1,0,0.0,0.00,1,0.11,1,2025-08-15T19:00:00Z,FWD,Bournemouth,100
784,Junior Kroupi,20,True,2,2,0,2.0,0.00,0,0.06,12,2025-08-23T14:00:00Z,FWD,Bournemouth,100
1343,Junior Kroupi,20,True,2,2,0,2.0,0.00,0,0.06,12,2025-08-23T14:00:00Z,FWD,Bournemouth,100
1491,Junior Kroupi,18,False,3,9,0,0.0,0.00,0,0.07,29,2025-08-30T14:00:00Z,FWD,Bournemouth,100
2056,Junior Kroupi,18,False,3,9,0,0.0,0.00,0,0.07,29,2025-08-30T14:00:00Z,FWD,Bournemouth,100
2205,Junior Kroupi,6,True,4,0,0,0.0,0.00,0,0.00,32,2025-09-13T14:00:00Z,FWD,Bournemouth,100
2796,Junior Kroupi,6,True,4,0,0,0.0,0.00,0,0.00,32,2025-09-13T14:00:00Z,FWD,Bournemouth,100


In [19]:
# Drop duplicate rows
match_logs = match_logs.drop_duplicates().reset_index(drop=True)
print(f"Exact duplicates: {match_logs.duplicated().sum():,}")

Exact duplicates: 0


In [20]:
# Double GW rows
match_logs[
    match_logs.duplicated(
        subset=['element', 'round'],
        keep=False
    )
]

,name,opponent_team,was_home,round,minutes,goals_scored,threat,expected_goals,goals_conceded,expected_goals_conceded,fixture,kickoff_time,position,team,element
19013,Adam Armstrong,16,False,26,90,0,2.0,0.00,0,2.55,257,2026-02-11T19:30:00Z,FWD,Wolves,817
19014,Adam Armstrong,1,True,26,90,0,1.0,0.03,2,1.86,310,2026-02-18T20:00:00Z,FWD,Wolves,817
19017,André Trindade da Costa Neto,16,False,26,0,0,0.0,0.00,0,0.00,257,2026-02-11T19:30:00Z,MID,Wolves,643
19018,André Trindade da Costa Neto,1,True,26,90,0,3.0,0.04,2,1.86,310,2026-02-18T20:00:00Z,MID,Wolves,643
19032,Myles Lewis-Skelly,5,False,26,0,0,0.0,0.00,0,0.00,252,2026-02-12T20:00:00Z,DEF,Arsenal,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27992,Ryan McAidoo,8,True,36,0,0,0.0,0.00,0,0.00,307,2026-05-13T19:00:00Z,MID,Man City,742
28001,Sverre Nypan,5,True,36,0,0,0.0,0.00,0,0.00,356,2026-05-09T16:30:00Z,MID,Man City,428
28002,Sverre Nypan,8,True,36,0,0,0.0,0.00,0,0.00,307,2026-05-13T19:00:00Z,MID,Man City,428
28061,Evann Guessand,9,True,36,0,0,0.0,0.00,0,0.00,353,2026-05-10T13:00:00Z,MID,Crystal Palace,677


In [21]:
dupes = (
    match_logs
    .groupby(['element', 'round'])
    .size()
    .reset_index(name='count')
)

dupes[dupes['count'] > 1]

,element,round,count
25,1,26,2
62,2,26,2
99,3,26,2
136,4,26,2
173,5,26,2
...,...,...,...
29080,809,26,2
29179,816,26,2
29192,817,26,2
29231,820,33,2


In [22]:
# Convert kickoff_time to datetime sortable value - to handle dgw potential
match_logs['kickoff_time'] = pd.to_datetime(match_logs['kickoff_time'])

In [23]:
# Sort values to ensure integrity of fixture order (dgw and kickoff_time)
match_logs = match_logs.sort_values(by=['element', 'round', 'kickoff_time'])

### Merge tables into one dataset for training

In [24]:
# Merge match_logs with fixture_data for upcoming gameweeks fixture difficulty.
# No common key between these exists (one uses team_id the other uses team name)
# merge team_data to match_logs first

In [25]:
match_logs.head()

,name,opponent_team,was_home,round,minutes,goals_scored,threat,expected_goals,goals_conceded,expected_goals_conceded,fixture,kickoff_time,position,team,element
519,David Raya Martín,14,False,1,90,0,0.0,0.0,0,1.52,9,2025-08-17 15:30:00+00:00,GK,Arsenal,1
1221,David Raya Martín,11,True,2,90,0,0.0,0.0,0,0.17,11,2025-08-23 16:30:00+00:00,GK,Arsenal,1
1931,David Raya Martín,12,False,3,90,0,0.0,0.0,1,0.52,25,2025-08-31 15:30:00+00:00,GK,Arsenal,1
2664,David Raya Martín,16,True,4,90,0,0.0,0.0,0,0.20,31,2025-09-13 11:30:00+00:00,GK,Arsenal,1
3405,David Raya Martín,13,True,5,90,0,0.0,0.0,1,0.89,41,2025-09-21 15:30:00+00:00,GK,Arsenal,1


In [26]:
fixture_data.head()

,event,id,team_a,team_h,team_h_difficulty,team_a_difficulty
0,1,1,4,12,3,4
1,1,2,15,2,3,4
2,1,3,10,6,3,4
3,1,6,3,18,1,3
4,1,5,19,17,2,3


In [27]:
team_data.head()

,name,id
0,Arsenal,1
1,Aston Villa,2
2,Burnley,3
3,Bournemouth,4
4,Brentford,5


In [28]:
# Merge team_data with match_logs data for team_id
main_df = pd.merge(
    match_logs,
    team_data,
    left_on = ["team"],
    right_on = ["name"]
)

In [29]:
# Clean up column names and drop duplicates
main_df = main_df.rename(columns={
    'name_x': 'player_name',
    'round': 'gameweek',
    'element': 'player_id',
    'id': 'team_id'
})

main_df = main_df.drop(["name_y"], axis=1)
main_df.columns

Index(['player_name', 'opponent_team', 'was_home', 'gameweek', 'minutes',
       'goals_scored', 'threat', 'expected_goals', 'goals_conceded',
       'expected_goals_conceded', 'fixture', 'kickoff_time', 'position',
       'team', 'player_id', 'team_id'],
      dtype='object')

In [30]:
# Check if in season transfers updates team and team_id
test_subset = main_df[main_df['player_name'].str.contains('semenyo', case=False, na=False)]

# Combine top 3 and bottom 3 rows
test_subset = pd.concat([test_subset.head(3), test_subset.tail(3)])
test_subset

,player_name,opponent_team,was_home,gameweek,minutes,goals_scored,threat,expected_goals,goals_conceded,expected_goals_conceded,fixture,kickoff_time,position,team,player_id,team_id
3078,Antoine Semenyo,12,False,1,90,2,32.0,0.91,4,2.21,1,2025-08-15 19:00:00+00:00,MID,Bournemouth,82,4
3079,Antoine Semenyo,20,True,2,90,0,45.0,0.68,0,0.46,12,2025-08-23 14:00:00+00:00,MID,Bournemouth,82,4
3080,Antoine Semenyo,18,False,3,90,0,30.0,0.21,0,0.19,29,2025-08-30 14:00:00+00:00,MID,Bournemouth,82,4
3113,Antoine Semenyo,8,True,36,90,1,31.0,0.23,0,0.68,307,2026-05-13 19:00:00+00:00,MID,Man City,82,13
3114,Antoine Semenyo,4,False,37,55,0,13.0,0.03,1,0.75,363,2026-05-19 18:30:00+00:00,MID,Man City,82,13
3115,Antoine Semenyo,2,True,38,57,1,39.0,0.10,1,0.45,376,2026-05-24 15:00:00+00:00,MID,Man City,82,13


#### Both team and team_id columns values change successfully

In [31]:
# Now we add fixture difficulty rating (FDR) data to our main dataframe
# team_h/a_difficulty (FDR) represents the perceived difficulty of the next fixture
# for that player rated 1-5 ordinally, with 1 being easiest and 5 being hardest

In [32]:
# Check for any 'duplicate' rows i.e. double gameweeks
home_duplicates = fixture_data.duplicated(subset=['event', 'team_h']).sum()
away_duplicates = fixture_data.duplicated(subset=['event', 'team_a']).sum()
print(f"Home duplicates: {home_duplicates}, Away duplicates: {away_duplicates}")

Home duplicates: 1, Away duplicates: 1


In [33]:
# Show double gameweek fixtures
dup_h_keys = fixture_data[fixture_data.duplicated(subset=["event", "team_h"], keep=False)]
dup_a_keys = fixture_data[fixture_data.duplicated(subset=["event", "team_a"], keep=False)]
print("Duplicated Fixture Key(s):")
print(dup_h_keys[["event", "team_h"]])
print(dup_a_keys[["event", "team_a"]])

Duplicated Fixture Key(s):
     event  team_h
353     36      13
359     36      13
     event  team_a
259     26       1
260     26       1


In [34]:
# Query df for home/away games only
home_df = main_df[main_df['was_home'] == True]
away_df = main_df[main_df['was_home'] == False]

In [35]:
# Check size of dfs
result = away_df.shape[0] + home_df.shape[0]
print(f"Away size: {away_df.shape[0]}, Home size: {home_df.shape[0]}, Total: {result}")

Away size: 14868, Home size: 14879, Total: 29747


In [36]:
# Merge home_df to fixture_data for fixture difficulty rating (FDR)
# Starting with home_df, we identify home games and assign the FDR of the home team

home_df = pd.merge(
    home_df,
    fixture_data[["team_a", "team_h", "team_h_difficulty"]],
    how = "left",
    left_on = ["opponent_team", "team_id"],
    right_on = ["team_a", "team_h"]
)

In [37]:
# Merge away_df to fixture_data for FDR

away_df = pd.merge(
    away_df,
    fixture_data[["team_h", "team_a", "team_a_difficulty"]],
    how = "left",
    left_on = ["opponent_team", "team_id"],
    right_on = ["team_h", "team_a"]
)

In [38]:
# Check size of df is the same after merge
result = away_df.shape[0] + home_df.shape[0]
print(f"Away size: {away_df.shape[0]}, Home size: {home_df.shape[0]}, Total: {result}")

Away size: 14868, Home size: 14879, Total: 29747


In [39]:
# Remove extra columns and change variable names
home_df = home_df.rename(columns={
    'team_h_difficulty': 'fdr'
})

away_df = away_df.rename(columns={
    'team_a_difficulty': 'fdr'
})

home_df = home_df.drop(['team_h', 'team_a'], axis=1)
away_df = away_df.drop(['team_a', 'team_h'], axis=1)

home_df.head()

,player_name,opponent_team,was_home,gameweek,minutes,goals_scored,threat,expected_goals,goals_conceded,expected_goals_conceded,fixture,kickoff_time,position,team,player_id,team_id,fdr
0,David Raya Martín,11,True,2,90,0,0.0,0.0,0,0.17,11,2025-08-23 16:30:00+00:00,GK,Arsenal,1,1,2
1,David Raya Martín,16,True,4,90,0,0.0,0.0,0,0.20,31,2025-09-13 11:30:00+00:00,GK,Arsenal,1,1,3
2,David Raya Martín,13,True,5,90,0,0.0,0.0,1,0.89,41,2025-09-21 15:30:00+00:00,GK,Arsenal,1,1,4
3,David Raya Martín,19,True,7,90,0,0.0,0.0,0,0.49,61,2025-10-04 14:00:00+00:00,GK,Arsenal,1,1,2
4,David Raya Martín,8,True,9,90,0,0.0,0.0,0,0.47,81,2025-10-26 14:00:00+00:00,GK,Arsenal,1,1,3


In [40]:
# FDR added, concatenate the home and away dfs together
main_df = pd.concat([home_df, away_df])

In [41]:
# Sort dataframe by team_id and gameweek
main_df = main_df.sort_values(by=['player_id', 'gameweek', 'team_id'])

In [42]:
main_df.head()

,player_name,opponent_team,was_home,gameweek,minutes,goals_scored,threat,expected_goals,goals_conceded,expected_goals_conceded,fixture,kickoff_time,position,team,player_id,team_id,fdr
0,David Raya Martín,14,False,1,90,0,0.0,0.0,0,1.52,9,2025-08-17 15:30:00+00:00,GK,Arsenal,1,1,4
0,David Raya Martín,11,True,2,90,0,0.0,0.0,0,0.17,11,2025-08-23 16:30:00+00:00,GK,Arsenal,1,1,2
1,David Raya Martín,12,False,3,90,0,0.0,0.0,1,0.52,25,2025-08-31 15:30:00+00:00,GK,Arsenal,1,1,4
1,David Raya Martín,16,True,4,90,0,0.0,0.0,0,0.20,31,2025-09-13 11:30:00+00:00,GK,Arsenal,1,1,3
2,David Raya Martín,13,True,5,90,0,0.0,0.0,1,0.89,41,2025-09-21 15:30:00+00:00,GK,Arsenal,1,1,4


### Feature Engineering

In [43]:
# Add aggregated goals/expected goals statistics and opposition stats for predictor variables
# Shift variables to avoid data leakage
# Remove useless variables
# Prepare dataset for training by encoding strings

#### Create attacking features - last game and last 3 games

In [44]:
# Function for previous games data

def previous_rolling_sum(x, interval):
    """Returns a rolling sum of previous rows based on interval"""
    return x.shift(1).rolling(interval, min_periods=1).sum()

In [45]:
# Create last game features for minutes, goals, expected goals, and threat
# Use groupby to ensure shift is locked within player_id

main_df['goals_last1'] = (
    main_df.groupby('player_id')['goals_scored']
        .transform(lambda x: previous_rolling_sum(x, 1))
)

main_df['mins_last1'] = (
    main_df.groupby('player_id')['minutes']
        .transform(lambda x: previous_rolling_sum(x, 1))
)

main_df['xg_last1'] = (
    main_df.groupby('player_id')['expected_goals']
        .transform(lambda x: previous_rolling_sum(x, 1))
)

main_df['threat_last1'] = (
    main_df.groupby('player_id')['threat']
        .transform(lambda x: previous_rolling_sum(x, 1))
)

In [46]:
# Create rolling previous 3 game statistics

main_df['goals_last3'] = (
    main_df.groupby('player_id')['goals_scored']
        .transform(lambda x: previous_rolling_sum(x, 3))
)

main_df['mins_last3'] = (
    main_df.groupby('player_id')['minutes']
        .transform(lambda x: previous_rolling_sum(x, 3))
)

main_df['xg_last3'] = (
    main_df.groupby('player_id')['expected_goals']
        .transform(lambda x: previous_rolling_sum(x, 3))
)

main_df['threat_last3'] = (
    main_df.groupby('player_id')['threat']
        .transform(lambda x: previous_rolling_sum(x, 3))
)

In [47]:
# Test cumulative stats - not including given gw data (shift(1))
main_df['season_mins'] = (
    main_df.groupby('player_id')['minutes']
        .transform(lambda x: x.shift(1).cumsum())
)

main_df['season_xg'] = (
    main_df.groupby('player_id')['expected_goals']
        .transform(lambda x: x.shift(1).cumsum())
)

main_df['season_goals'] = (
    main_df.groupby('player_id')['goals_scored']
        .transform(lambda x: x.shift(1).cumsum())
)

In [48]:
# Calculate xg_per90 feature - what about goals_per90?
# Avoid divide-by-zero
main_df['season_xg_per90'] = (
    main_df['season_xg'] / main_df['season_mins'].replace(0, np.nan) * 90
)


In [49]:
main_df.head()

,player_name,opponent_team,was_home,gameweek,minutes,goals_scored,threat,expected_goals,goals_conceded,expected_goals_conceded,fixture,kickoff_time,position,team,player_id,team_id,fdr,goals_last1,mins_last1,xg_last1,threat_last1,goals_last3,mins_last3,xg_last3,threat_last3,season_mins,season_xg,season_goals,season_xg_per90
0,David Raya Martín,14,False,1,90,0,0.0,0.0,0,1.52,9,2025-08-17 15:30:00+00:00,GK,Arsenal,1,1,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,David Raya Martín,11,True,2,90,0,0.0,0.0,0,0.17,11,2025-08-23 16:30:00+00:00,GK,Arsenal,1,1,2,0.0,90.0,0.0,0.0,0.0,90.0,0.0,0.0,90.0,0.0,0.0,0.0
1,David Raya Martín,12,False,3,90,0,0.0,0.0,1,0.52,25,2025-08-31 15:30:00+00:00,GK,Arsenal,1,1,4,0.0,90.0,0.0,0.0,0.0,180.0,0.0,0.0,180.0,0.0,0.0,0.0
1,David Raya Martín,16,True,4,90,0,0.0,0.0,0,0.20,31,2025-09-13 11:30:00+00:00,GK,Arsenal,1,1,3,0.0,90.0,0.0,0.0,0.0,270.0,0.0,0.0,270.0,0.0,0.0,0.0
2,David Raya Martín,13,True,5,90,0,0.0,0.0,1,0.89,41,2025-09-21 15:30:00+00:00,GK,Arsenal,1,1,4,0.0,90.0,0.0,0.0,0.0,270.0,0.0,0.0,360.0,0.0,0.0,0.0


In [50]:
main_df.columns

Index(['player_name', 'opponent_team', 'was_home', 'gameweek', 'minutes',
       'goals_scored', 'threat', 'expected_goals', 'goals_conceded',
       'expected_goals_conceded', 'fixture', 'kickoff_time', 'position',
       'team', 'player_id', 'team_id', 'fdr', 'goals_last1', 'mins_last1',
       'xg_last1', 'threat_last1', 'goals_last3', 'mins_last3', 'xg_last3',
       'threat_last3', 'season_mins', 'season_xg', 'season_goals',
       'season_xg_per90'],
      dtype='object')

#### Create historical features

In [51]:
# Add previous season goals, xg, and minutes
# Add aggregate mean goals and xg across all available seasons -- (per90)

player_history245.head()

,assists,bonus,bps,clean_sheets,clearances_blocks_interceptions,creativity,defensive_contribution,element_code,end_cost,expected_assists,expected_goal_involvements,expected_goals,expected_goals_conceded,goals_conceded,goals_scored,ict_index,influence,minutes,own_goals,penalties_missed,penalties_saved,recoveries,red_cards,saves,season_name,start_cost,starts,tackles,threat,total_points,yellow_cards,player_id,player_name
0,3,2,310,5,0,120.7,0,472713,49,0.58,0.96,0.38,28.13,30,0,45.1,227.4,1918,0,0,0,0,0,0,2022/23,50,23,0,102.0,63,7,116,Aaron_Hickey
1,0,0,87,2,0,28.4,0,472713,43,0.13,0.33,0.20,10.20,12,0,12.9,66.6,713,0,0,0,0,0,0,2023/24,45,9,0,37.0,17,5,116,Aaron_Hickey
2,0,0,0,0,0,0.0,0,472713,43,0.00,0.00,0.00,0.00,0,0,0.0,0.0,0,0,0,0,0,0,0,2024/25,45,0,0,0.0,0,0,116,Aaron_Hickey
3,0,0,0,0,0,0.0,0,225321,40,0.00,0.00,0.00,0.00,0,0,0.0,0.0,0,0,0,0,0,0,0,2016/17,40,0,0,0.0,0,0,674,Aaron_Ramsdale
4,0,0,0,0,0,0.0,0,225321,40,0.00,0.00,0.00,0.00,0,0,0.0,0.0,0,0,0,0,0,0,0,2017/18,40,0,0,0.0,0,0,674,Aaron_Ramsdale


In [52]:
player_history245.columns

Index(['assists', 'bonus', 'bps', 'clean_sheets',
       'clearances_blocks_interceptions', 'creativity',
       'defensive_contribution', 'element_code', 'end_cost',
       'expected_assists', 'expected_goal_involvements', 'expected_goals',
       'expected_goals_conceded', 'goals_conceded', 'goals_scored',
       'ict_index', 'influence', 'minutes', 'own_goals', 'penalties_missed',
       'penalties_saved', 'recoveries', 'red_cards', 'saves', 'season_name',
       'start_cost', 'starts', 'tackles', 'threat', 'total_points',
       'yellow_cards', 'player_id', 'player_name'],
      dtype='object')

In [53]:
# total variables are the sum of the players stats across all seasons up to and including the given season (24/25)
# last1 variables are the sum of the players stats from the most recent season in their dataset (mostly 24/25)

aggregated_history = player_history245.groupby('player_id').agg(
    total_goals = ('goals_scored', 'sum'),
    total_xg = ('expected_goals', 'sum'),
    total_mins = ('minutes', 'sum'),
    last_ssn_goals = ('goals_scored', 'last'),
    last_ssn_xg = ('expected_goals', 'last'),
    last_ssn_mins = ('minutes', 'last'),
    player_name = ('player_name', 'last')
).reset_index()

aggregated_history.head(10)

,player_id,total_goals,total_xg,total_mins,last_ssn_goals,last_ssn_xg,last_ssn_mins,player_name
0,1,0,0.11,11880,0,0.00,3420,David_Raya Martín
1,2,0,0.06,12510,0,0.06,2790,Kepa_Arrizabalaga Revuelta
2,3,0,0.00,0,0,0.00,0,Karl_Hein
3,4,0,0.00,0,0,0.00,0,Tommy_Setford
4,5,16,12.22,13873,3,2.70,2363,Gabriel_dos Santos Magalhães
5,6,6,5.03,8874,2,2.42,3039,William_Saliba
6,7,2,0.86,978,2,0.86,978,Riccardo_Calafiori
7,8,1,1.23,2485,1,1.23,2415,Jurriën_Timber
8,9,3,1.01,2484,1,0.17,1117,Jakub_Kiwior
9,10,1,0.16,1362,1,0.16,1362,Myles_Lewis-Skelly


In [54]:
# Create per90 statistics from historic statistics

aggregated_history['last_ssn_xgper90'] = (
    aggregated_history['last_ssn_xg'] / aggregated_history['last_ssn_mins'].replace(0, np.nan) * 90
)

aggregated_history['last_ssn_glsper90'] = (
    aggregated_history['last_ssn_goals'] / aggregated_history['last_ssn_mins'].replace(0, np.nan) * 90
)

aggregated_history['career_xgper90'] = (
    aggregated_history['total_xg'] / aggregated_history['total_mins'].replace(0, np.nan) * 90
)

aggregated_history['career_glsper90'] = (
    aggregated_history['total_goals'] / aggregated_history['total_mins'].replace(0, np.nan) * 90
)

In [55]:
aggregated_history.head(20)

,player_id,total_goals,total_xg,total_mins,last_ssn_goals,last_ssn_xg,last_ssn_mins,player_name,last_ssn_xgper90,last_ssn_glsper90,career_xgper90,career_glsper90
0,1,0,0.11,11880,0,0.00,3420,David_Raya Martín,0.000000,0.000000,0.000833,0.000000
1,2,0,0.06,12510,0,0.06,2790,Kepa_Arrizabalaga Revuelta,0.001935,0.000000,0.000432,0.000000
2,3,0,0.00,0,0,0.00,0,Karl_Hein,NaN,NaN,NaN,NaN
3,4,0,0.00,0,0,0.00,0,Tommy_Setford,NaN,NaN,NaN,NaN
4,5,16,12.22,13873,3,2.70,2363,Gabriel_dos Santos Magalhães,0.102835,0.114262,0.079276,0.103799
5,6,6,5.03,8874,2,2.42,3039,William_Saliba,0.071668,0.059230,0.051014,0.060852
6,7,2,0.86,978,2,0.86,978,Riccardo_Calafiori,0.079141,0.184049,0.079141,0.184049
7,8,1,1.23,2485,1,1.23,2415,Jurriën_Timber,0.045839,0.037267,0.044547,0.036217
8,9,3,1.01,2484,1,0.17,1117,Jakub_Kiwior,0.013697,0.080573,0.036594,0.108696
9,10,1,0.16,1362,1,0.16,1362,Myles_Lewis-Skelly,0.010573,0.066079,0.010573,0.066079


In [56]:
# Merge historic stats onto main training set

cols_to_keep = [
    "player_id",
    "last_ssn_goals",
    "last_ssn_xg",
    "last_ssn_mins",
    "last_ssn_xgper90",
    "last_ssn_glsper90",
    "career_xgper90",
    "career_glsper90"]

main_df = pd.merge(
    main_df,
    aggregated_history[cols_to_keep],
    how = "left",
    left_on = ["player_id"],
    right_on = ["player_id"]
)

main_df.head()

,player_name,opponent_team,was_home,gameweek,minutes,goals_scored,threat,expected_goals,goals_conceded,expected_goals_conceded,fixture,kickoff_time,position,team,player_id,team_id,fdr,goals_last1,mins_last1,xg_last1,threat_last1,goals_last3,mins_last3,xg_last3,threat_last3,season_mins,season_xg,season_goals,season_xg_per90,last_ssn_goals,last_ssn_xg,last_ssn_mins,last_ssn_xgper90,last_ssn_glsper90,career_xgper90,career_glsper90
0,David Raya Martín,14,False,1,90,0,0.0,0.0,0,1.52,9,2025-08-17 15:30:00+00:00,GK,Arsenal,1,1,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,3420.0,0.0,0.0,0.000833,0.0
1,David Raya Martín,11,True,2,90,0,0.0,0.0,0,0.17,11,2025-08-23 16:30:00+00:00,GK,Arsenal,1,1,2,0.0,90.0,0.0,0.0,0.0,90.0,0.0,0.0,90.0,0.0,0.0,0.0,0.0,0.0,3420.0,0.0,0.0,0.000833,0.0
2,David Raya Martín,12,False,3,90,0,0.0,0.0,1,0.52,25,2025-08-31 15:30:00+00:00,GK,Arsenal,1,1,4,0.0,90.0,0.0,0.0,0.0,180.0,0.0,0.0,180.0,0.0,0.0,0.0,0.0,0.0,3420.0,0.0,0.0,0.000833,0.0
3,David Raya Martín,16,True,4,90,0,0.0,0.0,0,0.20,31,2025-09-13 11:30:00+00:00,GK,Arsenal,1,1,3,0.0,90.0,0.0,0.0,0.0,270.0,0.0,0.0,270.0,0.0,0.0,0.0,0.0,0.0,3420.0,0.0,0.0,0.000833,0.0
4,David Raya Martín,13,True,5,90,0,0.0,0.0,1,0.89,41,2025-09-21 15:30:00+00:00,GK,Arsenal,1,1,4,0.0,90.0,0.0,0.0,0.0,270.0,0.0,0.0,360.0,0.0,0.0,0.0,0.0,0.0,3420.0,0.0,0.0,0.000833,0.0


#### Create defensive fixture statistics

In [57]:
# Create opposing team rolling defence statistics
opp_df = main_df.copy()
opp_df = opp_df[['team_id', 'gameweek', 'goals_conceded', 'expected_goals_conceded']]

# Sort values before groupby
opp_df = opp_df.sort_values(['team_id', 'gameweek'])
opp_df.head()

,team_id,gameweek,goals_conceded,expected_goals_conceded
0,1,1,0,1.52
38,1,1,0,0.00
76,1,1,0,0.00
114,1,1,0,0.00
152,1,1,0,1.52


In [58]:
# Aggregate by max values to collapse into 1 row per team per gameweek
opp_df = (
    opp_df.groupby(['team_id', 'gameweek'], as_index=False)
          .agg({
              'goals_conceded': 'max',
              'expected_goals_conceded': 'max'
          })
)

In [59]:
opp_df.head()

,team_id,gameweek,goals_conceded,expected_goals_conceded
0,1,1,0,1.52
1,1,2,0,0.17
2,1,3,1,0.52
3,1,4,0,0.20
4,1,5,1,0.89


In [60]:
opp_df['gc_last1'] = (
    opp_df.groupby('team_id')['goals_conceded']
    .transform(lambda x: previous_rolling_sum(x, 1))
)

opp_df['gc_last3'] = (
    opp_df.groupby('team_id')['goals_conceded']
    .transform(lambda x: previous_rolling_sum(x, 3))
)

opp_df['xgc_last1'] = (
    opp_df.groupby('team_id')['expected_goals_conceded']
    .transform(lambda x: previous_rolling_sum(x, 1))
)

opp_df['xgc_last3'] = (
    opp_df.groupby('team_id')['expected_goals_conceded']
    .transform(lambda x: previous_rolling_sum(x, 3))
)

In [61]:
opp_df['season_xgc'] = (
    opp_df.groupby('team_id')['expected_goals_conceded']
        .transform(lambda x: x.shift(1).cumsum())
)

opp_df['season_gc'] = (
    opp_df.groupby('team_id')['goals_conceded']
        .transform(lambda x: x.shift(1).cumsum())
)

In [62]:
opp_df.head(10)

,team_id,gameweek,goals_conceded,expected_goals_conceded,gc_last1,gc_last3,xgc_last1,xgc_last3,season_xgc,season_gc
0,1,1,0,1.52,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,0,0.17,0.0,0.0,1.52,1.52,1.52,0.0
2,1,3,1,0.52,0.0,0.0,0.17,1.69,1.69,0.0
3,1,4,0,0.20,1.0,1.0,0.52,2.21,2.21,1.0
4,1,5,1,0.89,0.0,1.0,0.20,0.89,2.41,1.0
5,1,6,1,0.61,1.0,2.0,0.89,1.61,3.30,2.0
6,1,7,0,0.49,1.0,2.0,0.61,1.70,3.91,3.0
7,1,8,0,0.44,0.0,2.0,0.49,1.99,4.40,3.0
8,1,9,0,0.47,0.0,1.0,0.44,1.54,4.84,3.0
9,1,10,0,0.42,0.0,0.0,0.47,1.40,5.31,3.0


In [63]:
# Merge main dataframe and opponent dataframe with keys next_opp and gameweek

main_df = main_df.merge(
    opp_df[['team_id','gameweek','gc_last1', 'gc_last3', 'xgc_last1', 'xgc_last3', 'season_xgc', 'season_gc']],
    how='left',
    left_on=['opponent_team','gameweek'],
    right_on=['team_id','gameweek'],
    suffixes=('','_opp')
)

In [64]:
main_df.head()

,player_name,opponent_team,was_home,gameweek,minutes,goals_scored,threat,expected_goals,goals_conceded,expected_goals_conceded,fixture,kickoff_time,position,team,player_id,team_id,fdr,goals_last1,mins_last1,xg_last1,threat_last1,goals_last3,mins_last3,xg_last3,threat_last3,season_mins,season_xg,season_goals,season_xg_per90,last_ssn_goals,last_ssn_xg,last_ssn_mins,last_ssn_xgper90,last_ssn_glsper90,career_xgper90,career_glsper90,team_id_opp,gc_last1,gc_last3,xgc_last1,xgc_last3,season_xgc,season_gc
0,David Raya Martín,14,False,1,90,0,0.0,0.0,0,1.52,9,2025-08-17 15:30:00+00:00,GK,Arsenal,1,1,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,3420.0,0.0,0.0,0.000833,0.0,14,NaN,NaN,NaN,NaN,NaN,NaN
1,David Raya Martín,11,True,2,90,0,0.0,0.0,0,0.17,11,2025-08-23 16:30:00+00:00,GK,Arsenal,1,1,2,0.0,90.0,0.0,0.0,0.0,90.0,0.0,0.0,90.0,0.0,0.0,0.0,0.0,0.0,3420.0,0.0,0.0,0.000833,0.0,11,0.0,0.0,0.55,0.55,0.55,0.0
2,David Raya Martín,12,False,3,90,0,0.0,0.0,1,0.52,25,2025-08-31 15:30:00+00:00,GK,Arsenal,1,1,4,0.0,90.0,0.0,0.0,0.0,180.0,0.0,0.0,180.0,0.0,0.0,0.0,0.0,0.0,3420.0,0.0,0.0,0.000833,0.0,12,2.0,4.0,0.93,2.63,2.63,4.0
3,David Raya Martín,16,True,4,90,0,0.0,0.0,0,0.20,31,2025-09-13 11:30:00+00:00,GK,Arsenal,1,1,3,0.0,90.0,0.0,0.0,0.0,270.0,0.0,0.0,270.0,0.0,0.0,0.0,0.0,0.0,3420.0,0.0,0.0,0.000833,0.0,16,3.0,5.0,2.42,5.02,5.02,5.0
4,David Raya Martín,13,True,5,90,0,0.0,0.0,1,0.89,41,2025-09-21 15:30:00+00:00,GK,Arsenal,1,1,4,0.0,90.0,0.0,0.0,0.0,270.0,0.0,0.0,360.0,0.0,0.0,0.0,0.0,0.0,3420.0,0.0,0.0,0.000833,0.0,13,0.0,4.0,1.51,4.92,5.48,4.0


In [65]:
main_df.columns

Index(['player_name', 'opponent_team', 'was_home', 'gameweek', 'minutes',
       'goals_scored', 'threat', 'expected_goals', 'goals_conceded',
       'expected_goals_conceded', 'fixture', 'kickoff_time', 'position',
       'team', 'player_id', 'team_id', 'fdr', 'goals_last1', 'mins_last1',
       'xg_last1', 'threat_last1', 'goals_last3', 'mins_last3', 'xg_last3',
       'threat_last3', 'season_mins', 'season_xg', 'season_goals',
       'season_xg_per90', 'last_ssn_goals', 'last_ssn_xg', 'last_ssn_mins',
       'last_ssn_xgper90', 'last_ssn_glsper90', 'career_xgper90',
       'career_glsper90', 'team_id_opp', 'gc_last1', 'gc_last3', 'xgc_last1',
       'xgc_last3', 'season_xgc', 'season_gc'],
      dtype='object')

In [66]:
# Drop unneccessary features

main_df = main_df.drop(columns=[
    'minutes',
    'threat',
    'expected_goals',
    'goals_conceded',
    'expected_goals_conceded',
    'team',
    'team_id',
    'team_id_opp'
])

main_df.head()

,player_name,opponent_team,was_home,gameweek,goals_scored,fixture,kickoff_time,position,player_id,fdr,goals_last1,mins_last1,xg_last1,threat_last1,goals_last3,mins_last3,xg_last3,threat_last3,season_mins,season_xg,season_goals,season_xg_per90,last_ssn_goals,last_ssn_xg,last_ssn_mins,last_ssn_xgper90,last_ssn_glsper90,career_xgper90,career_glsper90,gc_last1,gc_last3,xgc_last1,xgc_last3,season_xgc,season_gc
0,David Raya Martín,14,False,1,0,9,2025-08-17 15:30:00+00:00,GK,1,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,3420.0,0.0,0.0,0.000833,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,David Raya Martín,11,True,2,0,11,2025-08-23 16:30:00+00:00,GK,1,2,0.0,90.0,0.0,0.0,0.0,90.0,0.0,0.0,90.0,0.0,0.0,0.0,0.0,0.0,3420.0,0.0,0.0,0.000833,0.0,0.0,0.0,0.55,0.55,0.55,0.0
2,David Raya Martín,12,False,3,0,25,2025-08-31 15:30:00+00:00,GK,1,4,0.0,90.0,0.0,0.0,0.0,180.0,0.0,0.0,180.0,0.0,0.0,0.0,0.0,0.0,3420.0,0.0,0.0,0.000833,0.0,2.0,4.0,0.93,2.63,2.63,4.0
3,David Raya Martín,16,True,4,0,31,2025-09-13 11:30:00+00:00,GK,1,3,0.0,90.0,0.0,0.0,0.0,270.0,0.0,0.0,270.0,0.0,0.0,0.0,0.0,0.0,3420.0,0.0,0.0,0.000833,0.0,3.0,5.0,2.42,5.02,5.02,5.0
4,David Raya Martín,13,True,5,0,41,2025-09-21 15:30:00+00:00,GK,1,4,0.0,90.0,0.0,0.0,0.0,270.0,0.0,0.0,360.0,0.0,0.0,0.0,0.0,0.0,3420.0,0.0,0.0,0.000833,0.0,0.0,4.0,1.51,4.92,5.48,4.0


In [67]:
# Change position into numerical values with ordinal encoding

order = ['GK', 'DEF', 'MID', 'FWD']

encoder = OrdinalEncoder(categories=[order])

main_df['position'] = encoder.fit_transform(main_df[['position']])

In [68]:
# Change goals scored to binary feature for classification
main_df['goal_scored'] = np.where(main_df['goals_scored'] >= 1, 1, 0)

In [69]:
main_df.head()

,player_name,opponent_team,was_home,gameweek,goals_scored,fixture,kickoff_time,position,player_id,fdr,goals_last1,mins_last1,xg_last1,threat_last1,goals_last3,mins_last3,xg_last3,threat_last3,season_mins,season_xg,season_goals,season_xg_per90,last_ssn_goals,last_ssn_xg,last_ssn_mins,last_ssn_xgper90,last_ssn_glsper90,career_xgper90,career_glsper90,gc_last1,gc_last3,xgc_last1,xgc_last3,season_xgc,season_gc,goal_scored
0,David Raya Martín,14,False,1,0,9,2025-08-17 15:30:00+00:00,0.0,1,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,3420.0,0.0,0.0,0.000833,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0
1,David Raya Martín,11,True,2,0,11,2025-08-23 16:30:00+00:00,0.0,1,2,0.0,90.0,0.0,0.0,0.0,90.0,0.0,0.0,90.0,0.0,0.0,0.0,0.0,0.0,3420.0,0.0,0.0,0.000833,0.0,0.0,0.0,0.55,0.55,0.55,0.0,0
2,David Raya Martín,12,False,3,0,25,2025-08-31 15:30:00+00:00,0.0,1,4,0.0,90.0,0.0,0.0,0.0,180.0,0.0,0.0,180.0,0.0,0.0,0.0,0.0,0.0,3420.0,0.0,0.0,0.000833,0.0,2.0,4.0,0.93,2.63,2.63,4.0,0
3,David Raya Martín,16,True,4,0,31,2025-09-13 11:30:00+00:00,0.0,1,3,0.0,90.0,0.0,0.0,0.0,270.0,0.0,0.0,270.0,0.0,0.0,0.0,0.0,0.0,3420.0,0.0,0.0,0.000833,0.0,3.0,5.0,2.42,5.02,5.02,5.0,0
4,David Raya Martín,13,True,5,0,41,2025-09-21 15:30:00+00:00,0.0,1,4,0.0,90.0,0.0,0.0,0.0,270.0,0.0,0.0,360.0,0.0,0.0,0.0,0.0,0.0,3420.0,0.0,0.0,0.000833,0.0,0.0,4.0,1.51,4.92,5.48,4.0,0


In [70]:
def save_csv(df, name, current_time=None):
    filepath = os.path.join("data", "csv", name, f"{name}_{current_time}.csv")
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    df.to_csv(filepath, index=False)
    print(f"Saved {filepath}")

In [74]:
# Save copy of dataset
save_csv(main_df, "dataset_25_6")

Saved data\csv\dataset_25_6\dataset_25_6_None.csv
